In [8]:
import os
import gc
import time
import statistics as stats
from dataclasses import dataclass
from typing import Callable, List, Tuple, Dict, Any, Sequence, Union, Optional, Iterable
from pathlib import Path
from __future__ import annotations

import cv2
import numpy as np
import torch
import onnxruntime as ort

from SSD_from_scratch import mySSD
import CarImageClass
from SSDInt8_ONNX_Pred import SSDInt8ONNXPredictor, PreprocessConfig

# desktop or laptop
machine = 'laptop'

# Setup path to data folder
if machine == 'laptop':
    folder_path = Path(r"C:\self-driving-car\data")
else:
    folder_path = Path(r"C:\Udacity_car_data\data")

train_path = folder_path / "train"
test_path = folder_path / "test"



# os.environ["OMP_NUM_THREADS"] = "8"
# os.environ["MKL_NUM_THREADS"] = "8"

torch.set_num_threads(8)
torch.set_num_interop_threads(1)

RuntimeError: Error: cannot set number of interop threads after parallel work has started or set_num_interop_threads called

In [5]:
def _ms(ns: int) -> float:
    return ns / 1e6

def _pct(sorted_vals: List[float], p: float) -> float:
    # p in [0,100]
    if not sorted_vals:
        return float("nan")
    k = (len(sorted_vals) - 1) * (p / 100.0)
    f = int(k)
    c = min(f + 1, len(sorted_vals) - 1)
    if f == c:
        return sorted_vals[f]
    return sorted_vals[f] + (k - f) * (sorted_vals[c] - sorted_vals[f])

@dataclass
class StageStats:
    n: int
    mean_ms: float
    median_ms: float
    p90_ms: float
    p95_ms: float
    p99_ms: float
    min_ms: float
    max_ms: float

def summarize(times_ms: List[float]) -> StageStats:
    s = sorted(times_ms)
    return StageStats(
        n=len(s),
        mean_ms=sum(s) / len(s),
        median_ms=_pct(s, 50),
        p90_ms=_pct(s, 90),
        p95_ms=_pct(s, 95),
        p99_ms=_pct(s, 99),
        min_ms=s[0],
        max_ms=s[-1],
    )

def bench_inference(
    inputs: Iterable[Any],
    preprocess: Callable[[Any], Any],
    model: mySSD,
    warmup_iters: int = 20,
    measure_iters: int = 200,
) -> Dict[str, StageStats]:
    """
    CPU-only stage timing with warmup + per-iteration latency distributions.

    inputs: iterable of already-in-memory inputs (e.g., numpy arrays or decoded images)
    preprocess: input -> model_input
    forward: model_input -> raw_output
    postprocess: raw_output -> final_output
    """
    inputs = list(inputs)
    if not inputs:
        raise ValueError("inputs must be non-empty (and already in memory).")

    # --- Warmup (stabilizes caches, allocators, thread pools) ---
    wi = 0
    while wi < warmup_iters:
        x = inputs[wi % len(inputs)]
        mi = preprocess(x)
        loc_all, conf_all = model(mi) # forward(mi)
        _ = model.predict(x, score_thresh=0.3, nms_thresh=0.5, max_per_img=50, pre_loc_all=loc_all, pre_conf_all=conf_all) # postprocess(ro)
        wi += 1

    pre_t, fwd_t, post_t, e2e_t = [], [], [], []

    # --- Measure ---
    for i in range(measure_iters):
        x = inputs[i % len(inputs)]

        t0 = time.perf_counter_ns()
        mi = preprocess(x)
        t1 = time.perf_counter_ns()
        loc_all, conf_all = model(mi) # forward(mi)
        t2 = time.perf_counter_ns()
        _ = model.predict(x, score_thresh=0.3, nms_thresh=0.5, max_per_img=50, pre_loc_all=loc_all, pre_conf_all=conf_all) # postprocess(ro)
        t3 = time.perf_counter_ns()

        pre_t.append(_ms(t1 - t0))
        fwd_t.append(_ms(t2 - t1))
        post_t.append(_ms(t3 - t2))
        e2e_t.append(_ms(t3 - t0))

    out = {
        "preprocess": summarize(pre_t),
        "forward": summarize(fwd_t),
        "postprocess": summarize(post_t),
        "end_to_end": summarize(e2e_t),
    }

    # Sanity check: end_to_end should roughly equal sum of stages
    # If not, you probably have hidden work outside the stage calls (or timing overhead dominates).
    sum_means = out["preprocess"].mean_ms + out["forward"].mean_ms + out["postprocess"].mean_ms
    if abs(out["end_to_end"].mean_ms - sum_means) / max(out["end_to_end"].mean_ms, 1e-9) > 0.05:
        print(f"[warn] end_to_end mean ({out['end_to_end'].mean_ms:.3f} ms) != sum of means ({sum_means:.3f} ms).")
        print("       This usually indicates hidden work, extra copies, or stage boundary leakage.")

    return out

def print_report(report: Dict[str, StageStats]) -> None:
    for name, s in report.items():
        print(
            f"{name:>11}: n={s.n:4d}  mean={s.mean_ms:8.3f}  "
            f"p50={s.median_ms:8.3f}  p95={s.p95_ms:8.3f}  p99={s.p99_ms:8.3f}  "
            f"min={s.min_ms:8.3f}  max={s.max_ms:8.3f}"
        )

In [12]:
# for loading images

def load_images_uint8_chw(
    folder: str,
    *,
    exts=(".jpg", ".jpeg", ".png"),
    limit: int | None = None,
    convert_to_rgb: bool = False,   # keep False if your preprocessing expects BGR
) -> list[np.ndarray]:
    folder = Path(folder)
    paths = sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])
    if limit is not None:
        paths = paths[:limit]

    imgs: list[np.ndarray] = []
    for p in paths:
        im = cv2.imread(str(p), cv2.IMREAD_COLOR)  # uint8 HWC, BGR
        if im is None:
            raise ValueError(f"Failed to read: {p}")
        if convert_to_rgb:
            im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        # enforce contiguous memory (avoids occasional stride surprises)
        im = np.ascontiguousarray(np.transpose(im, (2, 0, 1)))
        imgs.append(im)

    if not imgs:
        raise ValueError(f"No images found in {folder} with extensions {exts}")
    return imgs

In [6]:
# preprocessing function

def preprocess_uint8_chw_rgb(
    x_chw: Union[np.ndarray, torch.Tensor],
    *,
    out_size: Tuple[int, int] = (300, 300),
    mean: Sequence[float] = (0.485, 0.456, 0.406),
    std: Sequence[float]  = (0.229, 0.224, 0.225),
) -> torch.Tensor:
    """
    Input:
        x_chw: uint8 (C,H,W), RGB, values in [0,255]
    Output:
        torch.float32 (1,C,out_H,out_W)

    Equivalent to:
        v2.ToImage()
        v2.ToDtype(torch.float32, scale=True)
        v2.Resize((300,300), antialias=True)
        v2.Normalize(mean=..., std=...)
    """
    # Convert to torch.Tensor
    if isinstance(x_chw, np.ndarray):
        x = torch.from_numpy(x_chw)
    elif isinstance(x_chw, torch.Tensor):
        x = x_chw
    else:
        raise TypeError(f"Expected numpy.ndarray or torch.Tensor, got {type(x_chw)}")

    if x.ndim != 3:
        raise ValueError(f"Expected shape (C,H,W), got {tuple(x.shape)}")
    if x.dtype != torch.uint8:
        raise TypeError(f"Expected dtype uint8, got {x.dtype}")

    # ToDtype(float32, scale=True) for uint8 => /255
    x = x.to(torch.float32) / 255.0

    # Add batch dim: (1,C,H,W)
    x = x.unsqueeze(0)

    # Resize to (300,300) with antialiasing (bilinear like torchvision for tensors)
    try:
        x = torch.nn.functional.interpolate(x, size=out_size, mode="bilinear", align_corners=False, antialias=True)
    except TypeError:
        # Fallback if antialias not supported in your PyTorch version
        x = torch.nn.functional.interpolate(x, size=out_size, mode="bilinear", align_corners=False)

    # Normalize
    mean_t = torch.tensor(mean, dtype=torch.float32, device=x.device).view(1, -1, 1, 1)
    std_t  = torch.tensor(std,  dtype=torch.float32, device=x.device).view(1, -1, 1, 1)
    x = (x - mean_t) / std_t

    return x

In [13]:
# load PyTorch SSD model
ssd_model_noZO_BS = mySSD(class_to_idx_dict={'biker': 0, 'car': 1, 'pedestrian': 2, 'trafficLight': 3, 'truck': 4},
                          in_channels=3,
                          variances=(0.1, 0.2))
# WEIGHTS_PATH = r"C:\Users\eblac\Documents\GitHub\self-driving-car\app_files\saved_models\noZoomOut_Bootstrap.pth"
WEIGHTS_PATH = r"C:\Users\eblac\OneDrive\Documents\GitHub\self-driving-car\app_files\saved_models\noZoomOut_Bootstrap.pth"
state_dict = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=False)
ssd_model_noZO_BS.load_state_dict(state_dict, strict=False)
ssd_model_noZO_BS.to(device='cpu');


# load images
img_list = np.array(load_images_uint8_chw(folder=test_path, convert_to_rgb=True, limit=1000))

In [14]:
# Ensure these are in-memory images/arrays, not filenames:
inputs = img_list

# If PyTorch:
# model.eval()
# forward = lambda x: model(x)  (inside inference_mode in your pipeline)

report = bench_inference(
    inputs=inputs,
    preprocess=preprocess_uint8_chw_rgb,
    model=ssd_model_noZO_BS,
    warmup_iters=30,
    measure_iters=500,
)
print_report(report)

 preprocess: n= 500  mean=   1.949  p50=   1.746  p95=   3.036  p99=   5.262  min=   1.431  max=   6.127
    forward: n= 500  mean= 327.886  p50= 319.218  p95= 385.309  p99= 482.338  min= 288.417  max= 594.984
postprocess: n= 500  mean=   8.228  p50=   6.894  p95=  16.901  p99=  21.102  min=   1.523  max=  46.425
 end_to_end: n= 500  mean= 338.063  p50= 329.715  p95= 397.477  p99= 504.201  min= 295.572  max= 607.748
